In [4]:
# import standard libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# import custom libraries
import lib.database.DBInterface as db_interface
from lib.utils.file_io import *

from setting_for_sdm.path_setting import path_list
from setting_for_sdm.date_setting import Date_Setting
from setting_for_sdm.constants import CONSTANTS
import setting_for_sdm.config as conf

import matplotlib as mpl
import lib.visualization.plot_generator as PlotGen
import lib.visualization.font_setting as font_setting
mpl.rcParams['font.family'] = font_setting.init_font()




Helvetica /home/mghan/.fonts/Helvetica/Helvetica Oblique.ttf
Registered font name: Helvetica


In [5]:
def filter_df(df, list_, type):
    if type == 'tag':
        pattern = '|'.join(re.escape(f'<{tag}>') for tag in list_)
    else:
        pattern = '|'.join(re.escape(f'{topic}') for topic in list_)

    mask = df[type].astype(str).str.contains(pattern, regex=True)
    return df[mask]

In [6]:
top_topics = load_json('/mnt/hdd/mghan/so_difficultyXavailability/data/python_top_topics.json')
bot_topics = load_json('/mnt/hdd/mghan/so_difficultyXavailability/data/python_bot_topics.json')

topic_df = load_df('/mnt/hdd/mghan/so_data_availability/result/bert_based/run_id_200/data', ['id' , 'creationdate' , 'title', 'tags', 'body', 'Topic'])
topic_df['creationdate'] = pd.to_datetime(topic_df['creationdate'], format="mixed").dt.normalize()


filtered_top_topic_df = filter_df(topic_df, top_topics, 'Topic')
filtered_bot_topic_df = filter_df(topic_df, bot_topics, 'Topic')

100%|██████████| 68/68 [00:19<00:00,  3.53it/s]


In [8]:
top_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/python_top_tags.json')
bot_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/python_bot_tags.json')

tag_df = load_all_files('/mnt/hdd/mghan/so_difficulty_measure/data/public_for_260105/questions/python/2020to2025')
tag_df = pd.DataFrame(tag_df)

filtered_top_tag_df = filter_df(tag_df, top_tags, 'tags'    )
filtered_bot_tag_df = filter_df(tag_df, bot_tags, 'tags'    )

In [9]:
print(filtered_top_tag_df.shape)
print(filtered_top_topic_df.shape)

(635031, 5)
(655431, 7)


In [26]:
top_tag_df = filtered_top_tag_df.sort_values('creationdate').copy()

In [27]:
top_tag_df.reset_index(drop=True, inplace=True)

In [43]:
top_tag_df['rel_week'] = np.floor((pd.to_datetime(top_tag_df['creationdate'], format='mixed')- datetime(2022,11,30)).dt.days/7)
top_tag_df['creationdate'] = pd.to_datetime(top_tag_df['creationdate'], format='yyyy-mm-dd')


In [58]:
date_list = list(top_tag_df['creationdate'].unique())

In [59]:
seq_start = 5000000
db_if = db_interface.DBInterface()

for idx, date in enumerate(date_list):
    result = []
    tmp = top_tag_df[top_tag_df['creationdate'] == date]
    for row in tmp.itertuples(index=False):
        result.append([seq_start + idx, row.creationdate, row.id])
    sql = f'INSERT INTO tt_posts_difficulty_target  VALUES %s'
    db_if.execute_bulk_values(sql, result)     

Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed successfully.
Bulk insert executed